# 00 · Prepare data and weights (CPU)

Builds the two inputs every later teacher notebook uses, so GPU sessions never download anything:

1. **Wikipedia data:** the Jigsaw Toxic Comment Classification Challenge files → cleaned
   `wiki_train_raw.parquet` and `wiki_test_raw.parquet` (`id`, `text`, six int8 labels), plus
   `data_manifest.json` with row counts, empty-text counts, and SHA-256 per file. These are the
   same rows and ids as TFDS `wikipedia_toxicity_subtypes/EnglishSubtypes:0.3.1`, whose download
   source now returns 403.
2. **Model weights:** `Qwen/Qwen3-4B-Instruct-2507` at the pinned revision, with each file's
   SHA-256 checked against Hugging Face.

**Before the first run:** accept the rules at
https://www.kaggle.com/c/jigsaw-toxic-comment-classification-challenge/rules, then in this
notebook use **Add Input → Competitions** and add `jigsaw-toxic-comment-classification-challenge`.

**Session settings:** Accelerator **None** (CPU), Internet **On**. No secrets needed.

**Run with "Save Version → Save & Run All"**, so the output is kept. Then attach this notebook's
output as an input to the later notebooks (see the last cell).

In [ ]:
# Code version. Use a commit hash once this notebook's code is on main.
REPO_URL = "https://github.com/Nishaant-Soni/Distillate.git"
REPO_REF = "Nishaant/Teacher"

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/tmp/Distillate")   # outside /kaggle/working, so the repo is not saved as output
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--quiet", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--quiet", REPO_REF], check=True)
CODE_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
sys.path.insert(0, str(REPO_DIR / "Model-Pipeline"))
print("code_commit:", CODE_COMMIT)

In [ ]:
from src.teacher.config import LABELS, load_config, model_version
from src.teacher.data import (
    build_data_manifest,
    check_frame,
    file_sha256,
    find_competition_dir,
    read_competition_split,
    to_raw_frame,
)

cfg = load_config()
OUT = Path("/kaggle/working")
DATA_OUT = OUT / "wiki"
MODEL_OUT = OUT / "qwen3-4b-instruct-2507"
DATA_OUT.mkdir(parents=True, exist_ok=True)
print("model_version:", model_version(cfg))

## 1. Wikipedia data

In [ ]:
COMP_DIR = find_competition_dir("/kaggle/input")   # fails if the competition is not attached
print("competition folder:", COMP_DIR)

results = {}
frames = {}
for split in ("train", "test"):
    df = to_raw_frame(read_competition_split(COMP_DIR, split))     # cleaning, 0/1 labels
    stats = check_frame(df, cfg["data"]["expected_rows"][split])   # row count, unique ids
    path = DATA_OUT / cfg["data"]["files"][split]
    df.to_parquet(path, index=False)
    results[path] = stats
    frames[split] = df
    print(split, stats)

source = {
    "kaggle_competition": cfg["data"]["kaggle_competition"],
    "files": {p.name: file_sha256(p) for p in sorted(COMP_DIR.iterdir()) if p.is_file()},
}
manifest_path = DATA_OUT / cfg["data"]["manifest"]
manifest_path.write_text(build_data_manifest(results, source, CODE_COMMIT))
print(manifest_path.read_text())

In [ ]:
# Sanity look: label prevalence (%) and text length per split. No comment text is printed.
import pandas as pd

prevalence = pd.DataFrame({split: df[list(LABELS)].mean() * 100 for split, df in frames.items()})
print(prevalence.round(2))
print(pd.DataFrame({split: df["text"].str.len().describe() for split, df in frames.items()})
      .round(0))

## 2. Model weights at the pinned revision

In [ ]:
import hashlib
import json
import shutil

from huggingface_hub import HfApi, snapshot_download

repo_id, revision = cfg["model"]["model_id"], cfg["model"]["revision"]
snapshot_download(repo_id=repo_id, revision=revision, local_dir=MODEL_OUT,
                  allow_patterns=["*.json", "*.safetensors", "*.txt", "LICENSE"])
shutil.rmtree(MODEL_OUT / ".cache", ignore_errors=True)   # download bookkeeping, not needed


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 24), b""):
            digest.update(chunk)
    return digest.hexdigest()


# Every large file must match the SHA-256 Hugging Face records for this revision.
info = HfApi().model_info(repo_id, revision=revision, files_metadata=True)
weights = {}
for sibling in info.siblings:
    local = MODEL_OUT / sibling.rfilename
    if sibling.lfs is None or not local.exists():
        continue
    actual = sha256(local)
    if actual != sibling.lfs.sha256:
        raise ValueError(f"checksum mismatch for {sibling.rfilename}")
    weights[sibling.rfilename] = {"bytes": local.stat().st_size, "sha256": actual}
if not any(name.endswith(".safetensors") for name in weights):
    raise ValueError("no safetensors files were downloaded")

(MODEL_OUT / "weights_manifest.json").write_text(json.dumps(
    {"model_id": repo_id, "revision": info.sha, "files": weights}, indent=2, sort_keys=True))
assert info.sha == revision, (info.sha, revision)
print(f"{len(weights)} weight files verified, "
      f"{sum(w['bytes'] for w in weights.values()) / 1e9:.2f} GB")

In [ ]:
# The tokenizer loads from the local copy and has single-token yes/no answers.
from transformers import AutoTokenizer

from src.teacher.scoring import answer_token_ids

tokenizer = AutoTokenizer.from_pretrained(MODEL_OUT)
yes_ids, no_ids = answer_token_ids(tokenizer)
print("YES_IDS", sorted(yes_ids), "NO_IDS", sorted(no_ids))

## 3. Acceptance and next step

The cells above raise an error if a check fails, so a finished run means:
* row counts are 159,571 (train) and 63,978 (test), ids are unique, and labels are 0/1;
* the empty-text count is recorded in `data_manifest.json`;
* the weights match Hugging Face's checksums for the pinned revision.

**Using the output:** in each later notebook, use **Add Input → Your Work → Notebooks** and pick
this notebook. Its `wiki/` and `qwen3-4b-instruct-2507/` folders then appear under `/kaggle/input/`.

In [ ]:
for path in sorted(OUT.rglob("*")):
    if path.is_file():
        print(f"{path.stat().st_size / 1e6:10.1f} MB  {path.relative_to(OUT)}")